In [1]:
from __future__ import annotations

import math
from typing import Dict

import torch
import torch.nn as nn
import torch.nn.functional as F


def _make_activation(name: str) -> nn.Module:
    name = str(name).strip().lower()
    if name == "tanh":
        return nn.Tanh()
    if name == "gelu":
        return nn.GELU()
    raise ValueError("activation must be 'tanh' or 'gelu'")


def _init_linear(layer: nn.Linear, gain: float = 1.0) -> None:
    nn.init.xavier_uniform_(layer.weight, gain=gain)
    nn.init.zeros_(layer.bias)


class ResidualBlock(nn.Module):
    def __init__(self, width: int, act: nn.Module):
        super().__init__()
        if width <= 0:
            raise ValueError("width must be > 0")

        self.fc1 = nn.Linear(width, width)
        self.fc2 = nn.Linear(width, width)
        self.act = act

        _init_linear(self.fc1)
        _init_linear(self.fc2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.act(self.fc1(x))
        h = self.fc2(h)
        return self.act(x + h)


class FlowPINNHardBC(nn.Module):
    """
    Input:
        X[:, 0] = x_nd
        X[:, 1] = y_nd
        X[:, 2] = Re_norm
        X[:, 3] = phi_nd

    Output:
        Y[:, 0] = u_nd
        Y[:, 1] = v_nd
        Y[:, 2] = p_nd

    Hard constraints:
        - u,v are gated by body SDF gate and wall gate
        - p is not gated
    """

    def __init__(
        self,
        y_bot: float,
        y_top: float,
        x_min: float,
        x_max: float,
        width: int = 128,
        n_blocks: int = 6,
        activation: str = "tanh",
        beta_body: float = 10.0,
        beta_wall: float = 10.0,
        softplus_k: float = 50.0,
        out_gain: float = 0.2,
    ):
        super().__init__()

        if not (y_top > y_bot):
            raise ValueError(f"Invalid walls: y_top={y_top} must be > y_bot={y_bot}")
        if not (x_max > x_min):
            raise ValueError(f"Invalid x bounds: x_max={x_max} must be > x_min={x_min}")
        if width <= 0:
            raise ValueError("width must be > 0")
        if n_blocks < 0:
            raise ValueError("n_blocks must be >= 0")
        if beta_body <= 0.0:
            raise ValueError("beta_body must be > 0")
        if beta_wall <= 0.0:
            raise ValueError("beta_wall must be > 0")
        if softplus_k <= 0.0:
            raise ValueError("softplus_k must be > 0")
        if out_gain <= 0.0:
            raise ValueError("out_gain must be > 0")

        self.act = _make_activation(activation)

        self.register_buffer("y_bot_buf", torch.tensor(float(y_bot), dtype=torch.float32))
        self.register_buffer("y_top_buf", torch.tensor(float(y_top), dtype=torch.float32))
        self.register_buffer("x_min_buf", torch.tensor(float(x_min), dtype=torch.float32))
        self.register_buffer("x_max_buf", torch.tensor(float(x_max), dtype=torch.float32))
        self.register_buffer("beta_body_buf", torch.tensor(float(beta_body), dtype=torch.float32))
        self.register_buffer("beta_wall_buf", torch.tensor(float(beta_wall), dtype=torch.float32))
        self.register_buffer("softplus_k_buf", torch.tensor(float(softplus_k), dtype=torch.float32))

        self.inp = nn.Linear(4, width)
        _init_linear(self.inp)

        self.blocks = nn.ModuleList(
            [ResidualBlock(width=width, act=self.act) for _ in range(n_blocks)]
        )

        self.out = nn.Linear(width, 3)
        _init_linear(self.out, gain=out_gain)

    @property
    def y_bot(self) -> float:
        return float(self.y_bot_buf.item())

    @property
    def y_top(self) -> float:
        return float(self.y_top_buf.item())

    @property
    def x_min(self) -> float:
        return float(self.x_min_buf.item())

    @property
    def x_max(self) -> float:
        return float(self.x_max_buf.item())

    def _scale_inputs(self, X: torch.Tensor) -> torch.Tensor:
        if X.ndim != 2 or X.shape[1] != 4:
            raise ValueError(f"Expected X shape [N,4], got {tuple(X.shape)}")

        Xs = X.clone()

        x = Xs[:, 0:1]
        y = Xs[:, 1:2]
        re_norm = Xs[:, 2:3]

        x_scaled = 2.0 * (x - self.x_min_buf) / (self.x_max_buf - self.x_min_buf + 1e-12) - 1.0
        y_scaled = 2.0 * (y - self.y_bot_buf) / (self.y_top_buf - self.y_bot_buf + 1e-12) - 1.0
        re_scaled = 2.0 * re_norm - 1.0

        Xs[:, 0:1] = x_scaled
        Xs[:, 1:2] = y_scaled
        Xs[:, 2:3] = re_scaled
        return Xs

    def _phi_pos_smooth(self, phi: torch.Tensor) -> torch.Tensor:
        """
        Smooth approximation of max(phi, 0), shifted so phi=0 -> 0 exactly.
        """
        k = self.softplus_k_buf.to(dtype=phi.dtype, device=phi.device)
        sp = F.softplus(k * phi)
        sp0 = math.log(2.0)
        phi_pos = (sp - sp0) / (k + 1e-12)
        return torch.clamp(phi_pos, min=0.0)

    def _gate_body(self, phi: torch.Tensor) -> torch.Tensor:
        phi_pos = self._phi_pos_smooth(phi)
        beta = self.beta_body_buf.to(dtype=phi.dtype, device=phi.device)
        return torch.tanh(beta * phi_pos)

    def _gate_wall(self, y: torch.Tensor) -> torch.Tensor:
        yb = self.y_bot_buf.to(dtype=y.dtype, device=y.device)
        yt = self.y_top_buf.to(dtype=y.dtype, device=y.device)

        d_bot = y - yb
        d_top = yt - y
        d = torch.minimum(d_bot, d_top)
        d = torch.clamp(d, min=0.0)

        half_height = 0.5 * (yt - yb)
        d_norm = d / (half_height + 1e-12)

        beta = self.beta_wall_buf.to(dtype=y.dtype, device=y.device)
        return torch.tanh(beta * d_norm)

    def compute_gates(self, X: torch.Tensor) -> Dict[str, torch.Tensor]:
        y = X[:, 1:2]
        phi = X[:, 3:4]

        s_body = self._gate_body(phi)
        s_wall = self._gate_wall(y)
        s_uv = s_body * s_wall

        return {
            "s_body": s_body,
            "s_wall": s_wall,
            "s_uv": s_uv,
        }

    def forward_raw(self, X: torch.Tensor) -> torch.Tensor:
        Xs = self._scale_inputs(X)

        h = self.act(self.inp(Xs))
        for block in self.blocks:
            h = block(h)

        return self.out(h)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        raw = self.forward_raw(X)
        gates = self.compute_gates(X)

        s_uv = gates["s_uv"]

        u = s_uv * raw[:, 0:1]
        v = s_uv * raw[:, 1:2]
        p = raw[:, 2:3]

        return torch.cat([u, v, p], dim=1)